# Tema: Auto Loader

## Objetivos
Ingerir JSON incremental, rescatar campos y practicar evolución de esquema.

## Conceptos importantes para el examen
cloudFiles; schemaLocation describe esquema y checkpointLocation describe progreso; schemaHints no equivale a schema fijo; addNewColumns exige reinicio al descubrir columnas nuevas.

**Dificultad:** Intermedio · **Tiempo estimado:** 75 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_16_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
import json
records = [{"event_id": i, "customer_id": i % 4, "amount": i * 10} for i in range(1, 13)]
# El directorio se crea en la siguiente celda; todavía no hay ninguna carga.

In [ ]:
# Requiere CREATE VOLUME en el schema; alternativa: usa un volumen autorizado.
spark.sql("CREATE VOLUME IF NOT EXISTS lab_files")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/lab_files"
dbutils.fs.mkdirs(BASE + "/landing")
CHECKPOINT = BASE + "/checkpoints/main"
print(BASE)
SOURCE = BASE + "/landing"
dbutils.fs.put(SOURCE + "/batch_01.json", "\n".join(json.dumps(r) for r in records), overwrite=False)

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Inferencia con hints y rescate
El directorio contiene datos antes de inferir. El modo rescue guarda campos nuevos sin evolucionar las columnas de salida.

In [ ]:
def rescue_stream():
    return (spark.readStream.format("cloudFiles").option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", BASE + "/schemas/rescue")
            .option("cloudFiles.schemaHints", "event_id INT, customer_id INT, amount DOUBLE")
            .option("cloudFiles.schemaEvolutionMode", "rescue")
            .option("rescuedDataColumn", "_rescued_data").load(SOURCE))

### 2. Carga incremental disponible

In [ ]:
def run_auto():
    q = (rescue_stream().writeStream.format("delta").option("checkpointLocation", CHECKPOINT)
         .trigger(availableNow=True).toTable("bronze_auto"))
    q.awaitTermination()
    return q
q = run_auto()
assert spark.table("bronze_auto").count() == 12

### 3. Descubrimiento y alcance
Este laboratorio usa listado del directorio; notificaciones/file events requieren configuración de la external location y soporte del entorno.

In [ ]:
q = run_auto()
assert spark.table("bronze_auto").count() == 12
display(dbutils.fs.ls(BASE + "/schemas/rescue"))
print(q.recentProgress)

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Añade un fichero con eventos 13 y 14 y ejecuta de nuevo. Verifica 14 filas.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Añade evento 15 con la columna extra channel. Observa _rescued_data.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Crea una fuente separada inicialmente sin channel y una carga addNewColumns. Prepara función run_evolve y ejecútala.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Añade channel a esa nueva fuente. Observa el fallo de nueva columna, revisa el error y reinicia con el mismo checkpoint.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Construye Silver tipada y cuarentena por importe nulo o negativo a partir de bronze_auto.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** Nombre nuevo, mismo checkpoint.

**Pista 2:** rescue no cambia el esquema de columnas.

**Pista 3:** schemaHints, schemaLocation y checkpoint nuevos; mergeSchema en destino.

**Pista 4:** Solo trata UnknownField como fallo esperado; no ocultes otros fallos.

**Pista 5:** Auto Loader ingiere archivos; la calidad de negocio es otro paso.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
dbutils.fs.put(SOURCE + "/batch_02.json", "\n".join(json.dumps({"event_id": i, "customer_id": 1, "amount": 50}) for i in [13,14]), overwrite=False)
q = run_auto()
assert spark.table("bronze_auto").count() == 14

### Solución 2

In [ ]:
dbutils.fs.put(SOURCE + "/batch_03.json", json.dumps({"event_id":15, "customer_id":1, "amount":60, "channel":"web"}), overwrite=False)
q = run_auto()
display(spark.table("bronze_auto").filter("_rescued_data IS NOT NULL"))

### Solución 3

In [ ]:
EVOLVE = BASE + "/evolve_input"
dbutils.fs.mkdirs(EVOLVE)
dbutils.fs.put(EVOLVE + "/a.json", json.dumps(records[0]), overwrite=False)
def run_evolve():
    stream = (spark.readStream.format("cloudFiles").option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", BASE + "/schemas/evolve")
        .option("cloudFiles.schemaHints", "event_id INT, customer_id INT, amount DOUBLE")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns").load(EVOLVE))
    q = (stream.writeStream.format("delta").option("mergeSchema", "true")
        .option("checkpointLocation", BASE + "/checkpoints/evolve")
        .trigger(availableNow=True).toTable("bronze_evolved"))
    q.awaitTermination()
run_evolve()

### Solución 4

In [ ]:
dbutils.fs.put(EVOLVE + "/b.json", json.dumps({"event_id": 50, "customer_id":2, "amount":80, "channel":"app"}), overwrite=False)
try:
    run_evolve()
except Exception as exc:
    message = str(exc)
    if not any(token in message.lower() for token in ["unknownfield", "unknown_field", "new fields", "new column"]):
        raise
    print("Evolución detectada; se reinicia con el esquema actualizado.")
    run_evolve()
assert "channel" in spark.table("bronze_evolved").columns
display(spark.table("bronze_evolved"))

### Solución 5

In [ ]:
bronze = spark.table("bronze_auto")
valid = (F.col("event_id").isNotNull() & F.col("amount").isNotNull() & (F.col("amount") >= 0))
bronze.filter(valid).write.format("delta").mode("overwrite").saveAsTable("silver_auto")
bronze.filter(~F.coalesce(valid, F.lit(False))).write.format("delta").mode("overwrite").saveAsTable("quarantine_auto")

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Dónde mantiene Auto Loader la evolución del esquema?

A. Solo en el nombre del job

B. cloudFiles.schemaLocation

C. ORDER BY

D. En un grupo de UC

### Pregunta 2
Con addNewColumns aparece una columna desconocida. ¿Qué comportamiento se espera?

A. Se descarta siempre sin aviso

B. Se cambia a CSV

C. Se actualiza el esquema y se requiere reinicio

D. Se borra el destino

### Pregunta 3
¿Qué opción permite sugerir tipos sin fijar un schema completo?

A. cloudFiles.schemaHints

B. checkpointLocation

C. outputMode

D. VACUUM

### Respuestas y explicación
**1. B** — Esa ruta almacena los esquemas inferidos.

**2. C** — El reinicio retoma con el esquema actualizado.

**3. A** — Los hints orientan la inferencia.

### Documentación oficial
- [Esquema de Auto Loader](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema)

## PARTE 6 - RETO FINAL
Recibe tres ficheros con evolución y un importe incorrecto. Conserva Bronze, separa cuarentena y justifica rescue frente a addNewColumns.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
